<a href="https://colab.research.google.com/github/antoine-bourgois/MIA_2026_UE16_NLP/blob/main/code/2_Cooccurrence_Tokens_Vecteurs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## TP 2 : Représentation Tokens - Matrice de cooccurrence

- Charger un Dataset
- Description statistique du dataset
- Analyse tokens vs lemmes
- Distribution des tokens

### Installation des librairies Python utiles pour le TP

In [1]:
!pip install tqdm jupyter ipywidgets datasets pyarrow huggingface_hub pandas matplotlib numpy scipy scikit-learn umap-learn plotly --quiet


[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: pip install --upgrade pip


In [2]:
from tqdm.auto import tqdm
from collections import Counter

### Charger un jeu de données

In [3]:
from datasets import load_dataset_builder

dataset_path = "AntoineBourgois/Gallica_Hors-d-Usage_cleaned"

builder = load_dataset_builder(dataset_path)

builder = load_dataset_builder(dataset_path, )

print(f"Le jeu de données {dataset_path!r} contient {len(builder.info.splits)} sous-ensembles :")
for name, split in builder.info.splits.items():
    print(f"    {name!r:<20}{split.num_examples:<6,} documents")

Le jeu de données 'AntoineBourgois/Gallica_Hors-d-Usage_cleaned' contient 5 sous-ensembles :
    'tous'              6,156  documents
    'auteur_genre'      200    documents
    'auteur'            200    documents
    'date_publication'  200    documents
    'genre_litteraire'  200    documents


In [4]:
# sous_ensemble, label_column = "auteur_genre", "auteur_genre"
# sous_ensemble, label_column = "auteur", "auteur"
sous_ensemble, label_column = "date_publication", "date_publication"
# sous_ensemble, label_column = "genre_litteraire", "genre_litteraire"

import pyarrow.parquet as pq
from huggingface_hub import hf_hub_download

dataset_path = "AntoineBourgois/Gallica_Hors-d-Usage_cleaned"

fichier_local = hf_hub_download(
    repo_id=dataset_path,
    filename=f"data/{sous_ensemble}-00000-of-00001.parquet",
    repo_type="dataset",
    force_download=True,
)

table = pq.read_table(fichier_local)
dataset = table.to_pylist()

print("Nombre de documents :", len(dataset))


def conversions_string_vers_liste(string):
    if isinstance(string, str):
        return string.split(" ")


for document in dataset:
    document["tokens"] = conversions_string_vers_liste(document["tokens"])
    document["lemmes"] = conversions_string_vers_liste(document["lemmes"])

data/date_publication-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B /  178MB            

data/date_publication-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Nombre de documents : 200


### Structure du dataset

In [5]:
print(f"Type dataset: {type(dataset)}")
print(f"Longueur dataset: {len(dataset)}")

doc_0 = dataset[0]
print(f"Type 1er document: {type(doc_0)}")
print(f"Clés d'un document: {doc_0.keys()}")

Type dataset: <class 'list'>
Longueur dataset: 200
Type 1er document: <class 'dict'>
Clés d'un document: dict_keys(['doc_id', 'date_publication', 'titre', 'auteur', 'auteur_genre', 'auteur_nom', 'auteur_prenom', 'genre_litteraire', 'texte', 'tokens', 'lemmes'])


In [6]:
for key, value in doc_0.items():
    print(f"{key:<20} {type(value)}    {str(value)[:100]}")

doc_id               <class 'str'>    HDU-00027
date_publication     <class 'int'>    1801
titre                <class 'str'>    Atala, ou les Amours de deux sauvages dans le désert
auteur               <class 'str'>    François-René de Chateaubriand
auteur_genre         <class 'str'>    masculin
auteur_nom           <class 'str'>    Chateaubriand
auteur_prenom        <class 'str'>    François-René de
genre_litteraire     <class 'str'>    autre
texte                <class 'str'>    Ans sur cette quatrième Edition'. -S quelque temps, il a paru de nouvelles critiques d'Atala. Je n'a
tokens               <class 'list'>    ['Ans', 'sur', 'cette', 'quatrième', 'Edition', "'", '.', '-S', 'quelque', 'temps', ',', 'il', 'a', 
lemmes               <class 'list'>    ['an', 'sur', 'ce', 'quatrième', 'edition', "'", '.', '-s', 'quelque', 'temps', ',', 'il', 'avoir', 


### Vocabulaire

Construire un dictionnaire du vocabulaire du corpus.

### Choix de l'unité élémentaire : tokens ou lemmes ?

In [7]:
for word_type in ["tokens", "lemmes"]:
    all_words = [word.lower()
                  for document in dataset
                  for word in document[word_type]
                  ]
    vocab = sorted(set(all_words))
    print(f"{word_type:<10}  Taille du vocabulaire: {len(vocab):,} words")

tokens      Taille du vocabulaire: 322,838 words
lemmes      Taille du vocabulaire: 285,868 words


Nous retenons les lemmes pour réduire la taille du vocabulaire

In [8]:
word_type = "lemmes" # ["lemmes", "tokens"]

In [9]:
all_words = [word.lower()
                  for document in dataset
                  for word in document[word_type]
                  ]
vocab = sorted(set(all_words))


print(f"Documents : {len(dataset):,}")
print(f"Taille du corpus : {len(all_words):,}")
print(f"Mots / Document : {int(len(all_words)/len(dataset)):,}")
print(f"Taille du vocabulaire : {len(vocab):,} ({word_type})")

Documents : 200
Taille du corpus : 19,719,818
Mots / Document : 98,599
Taille du vocabulaire : 285,868 (lemmes)


In [10]:
import pandas as pd
from collections import Counter

vocab_counter = Counter(all_words).most_common()
# vocab = list(dict(vocab_counter).keys())

df = pd.DataFrame([
    {
        "Rang": i + 1,
        "Word": word,
        "Occurrences": count,
        "Fréquence (%)": 100 * count / len(all_words)
    }
    for i, (word, count) in enumerate(vocab_counter)
])

df

,Rang,Word,Occurrences,Fréquence (%)
0,1,",",1337995,6.785027
1,2,le,1252190,6.349906
2,3,de,1067224,5.411936
3,4,.,693278,3.515641
4,5,un,434497,2.203352
...,...,...,...,...
285863,285864,mous,1,0.000005
285864,285865,silenciesuement,1,0.000005
285865,285866,téqui,1,0.000005
285866,285867,1948,1,0.000005


In [11]:
word_lists = [
    [word.lower() for word in doc[word_type]]
    for doc in dataset
]
word_counts = Counter(all_words)

print(f"Vocabulaire : {len(vocab_counter):,}")
print(f"Mots apparaissant une seule fois : {sum(c[1] == 1 for c in vocab_counter):,}")
print(f"Mots apparaissant 5 fois ou plus : {sum(c[1] >= 5 for c in vocab_counter):,}")
print(f"Mots apparaissant 20 fois ou plus : {sum(c[1] >= 20 for c in vocab_counter):,}")
print(f"Mots apparaissant 100 fois ou plus : {sum(c[1] >= 100 for c in vocab_counter):,}")
print(f"Mots apparaissant 1 000 fois ou plus : {sum(c[1] >= 1000 for c in vocab_counter):,}")

Vocabulaire : 285,868
Mots apparaissant une seule fois : 189,014
Mots apparaissant 5 fois ou plus : 43,203
Mots apparaissant 20 fois ou plus : 20,367
Mots apparaissant 100 fois ou plus : 7,926
Mots apparaissant 1 000 fois ou plus : 1,528


In [12]:
import numpy as np
from scipy.sparse import coo_matrix
from tqdm import tqdm

# Context Window
N = 3

word_to_id = {word: i for i, word in enumerate(vocab)}

rows = []
cols = []

for tokens in tqdm(word_lists):

    ids = np.fromiter(
        (word_to_id[w] for w in tokens if w in word_to_id),
        dtype=np.int32
    )

    # Generate context pairs for each distance
    for d in range(1, N + 1):

        if len(ids) <= d:
            continue

        left = ids[:-d]
        right = ids[d:]

        # Both directions: target -> context and context -> target
        rows.append(np.concatenate((left, right)))
        cols.append(np.concatenate((right, left)))

# Concatenate all pairs
rows = np.concatenate(rows)
cols = np.concatenate(cols)

# Build sparse matrix
cooc_matrix = coo_matrix(
    (
        np.ones(len(rows), dtype=np.int32),
        (rows, cols)
    ),
    shape=(len(vocab), len(vocab))
).tocsr()

print("Dimensions de la matrice :", cooc_matrix.shape)
print(f"Nombre d'entrées non nulles : {cooc_matrix.nnz:,} ({cooc_matrix.nnz/(cooc_matrix.shape[0]*cooc_matrix.shape[1]):.4%})")

100%|██████████| 200/200 [00:05<00:00, 37.56it/s]


Dimensions de la matrice : (285868, 285868)
Nombre d'entrées non nulles : 12,754,864 (0.0156%)


In [13]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def get_most_similar(
    word,
    matrix,
    word_to_id,
    vocab,
    word_counts,
    top_n=10,
    min_count=1
):

    if word not in word_to_id:
        print(f"'{word}' is not in the vocabulary.")
        return

    word_id = word_to_id[word]

    # Compute similarity against all word vectors
    similarities = cosine_similarity(
        matrix[word_id],
        matrix
    ).flatten().tolist()

    # Rank candidates
    closest_ids = np.argsort(similarities)[::-1]

    results = []

    for idx in closest_ids:

        candidate = vocab[idx]

        # Exclude the query word
        if idx == word_id:
            continue

        # Apply frequency threshold
        if word_counts[candidate] < min_count:
            continue

        results.append((candidate, round(similarities[idx], 4)))

        if len(results) == top_n:
            break

    return results

In [14]:
min_word_occurrences = 0

filtered_vocab = [
    word for word, count in word_counts.items()
    if count >= min_word_occurrences
]
print(f"Vocabulaire après filtrage: {len(filtered_vocab):,}")

Vocabulaire après filtrage: 285,868


In [148]:
target_word = "mère"
most_similar = get_most_similar(
    target_word,
    cooc_matrix,
    word_to_id,
    vocab,
    word_counts,
    top_n=500,
    min_count=min_word_occurrences
)
print(f"Mot Cible  : {target_word}")
print(f"Contexte N : {N}")
print(f"-----------------")
print(pd.DataFrame(most_similar, columns=["Word", "Cos_Sim"]).to_string(index=False, header=False, justify=["right", "center"]))

Mot Cible  : mère
Contexte N : 3
-----------------
          père 0.9910
         soeur 0.9900
         frère 0.9900
          sœur 0.9883
         nièce 0.9848
          fils 0.9845
         neveu 0.9804
          mari 0.9787
          amie 0.9752
      camarade 0.9735
     maîtresse 0.9732
         enfan 0.9728
        parent 0.9711
   imagination 0.9669
         élève 0.9657
    protecteur 0.9654
        cousin 0.9653
          hôte 0.9650
         amant 0.9604
         oncle 0.9601
        maître 0.9584
         époux 0.9580
          cœur 0.9576
         coeur 0.9575
        favori 0.9563
           ami 0.9547
         tante 0.9542
    conscience 0.9539
       enfance 0.9538
      démarche 0.9527
      jeunesse 0.9508
       fortune 0.9508
      complice 0.9506
     personnel 0.9506
        épouse 0.9506
         rival 0.9502
        pensée 0.9479
        client 0.9467
     compagnon 0.9451
      habituel 0.9451
        fidèle 0.9449
    adversaire 0.9445
         guide 0.9440
   

In [84]:
target_word = "joue"
most_similar = get_most_similar(
    target_word,
    cooc_matrix,
    word_to_id,
    vocab,
    word_counts,
    top_n=10,
    min_count=0
)
print(f"Mot Cible  : {target_word}")
print(f"Contexte N : {N}")
print(f"-----------------")
print(pd.DataFrame(most_similar, columns=["Word", "Cos_Sim"]).to_string(index=False, header=False, justify=["right", "center"]))

Mot Cible  : joue
Contexte N : 3
-----------------
   front 0.9818
   lèvre 0.9801
  visage 0.9786
poitrine 0.9761
   jambe 0.9735
   tempe 0.9658
  cheveu 0.9624
    oeil 0.9577
  cuisse 0.9556
  figure 0.9527


In [104]:
for target_word in ["lundi", "fusil", "joue", "maison", "roi", "tristesse", "sept", "france", "robe"]:
   most_similar = get_most_similar(
    target_word,
    cooc_matrix,
    word_to_id,
    vocab,
    word_counts,
    top_n=5,
    min_count=0)
   print(f"{target_word:<10} -> {most_similar[0][0]:<10} ->  {most_similar[1][0]:<10} ->  {most_similar[2][0]:<10} ->  {most_similar[3][0]:<10} ->  {most_similar[4][0]:<10}")

lundi      -> dimanche   ->  mardi      ->  vendredi   ->  jeudi      ->  samedi    
fusil      -> sabre      ->  bâton      ->  couteau    ->  carabine   ->  pistolet  
joue       -> front      ->  lèvre      ->  visage     ->  poitrine   ->  jambe     
maison     -> ville      ->  société    ->  grotte     ->  intérieur  ->  parc      
roi        -> gouverneur ->  chevalier  ->  ministre   ->  cardinal   ->  peuple    
tristesse  -> angoisse   ->  mélancolie ->  joie       ->  terreur    ->  embarras  
sept       -> neuf       ->  quatre     ->  dix        ->  cinq       ->  six       
france     -> bretagne   ->  europe     ->  angleterre ->  italie     ->  orient    
robe       -> tunique    ->  bonnet     ->  costume    ->  habit      ->  armure    


In [105]:
min_word_occurrences = 100

filtered_vocab = [
    word for word, count in word_counts.items()
    if count >= min_word_occurrences
]

print(f"Vocabulary after filtering: {len(filtered_vocab):,}")

Vocabulary after filtering: 7,926


In [106]:
# Original vocabulary indices of retained words
filtered_ids = [
    word_to_id[word]
    for word in filtered_vocab
]

# Filter rows AND columns
filtered_matrix = cooc_matrix[filtered_ids, :][:, filtered_ids]
filtered_matrix = filtered_matrix.astype("float32")

print("Original matrix:", cooc_matrix.shape)
print("Filtered matrix:", filtered_matrix.shape)

Original matrix: (285868, 285868)
Filtered matrix: (7926, 7926)


In [141]:
import umap

reducer = umap.UMAP(
    n_components=2,
    n_neighbors=25,
    min_dist=0.1,
    metric="cosine",
    n_jobs=-1,
    # random_state=42
)

embedding = reducer.fit_transform(filtered_matrix)

# 2. Mean-center the entire 3D projection around (0, 0, 0)
embedding = embedding - embedding.mean(axis=0)

/home/antoine/cours/DAUPHINE_2026-2027/.venv/lib/python3.12/site-packages/umap/umap_.py:4160: FutureWarning: compatibility_layout was not set, so UMAP is using the legacy (0.5.x) nearest-neighbor, initialization and layout pipeline. In a future release the default will change to compatibility_layout=False, which uses recursive initialization and the Adam optimizer and produces different embeddings. Set compatibility_layout=True to keep the current behavior and silence this warning, or compatibility_layout=False (or optimizer='adam') to adopt the new behavior now.
  self.fit(X, y, ensure_all_finite, **kwargs)


In [142]:
import pandas as pd
import plotly.express as px

embedding_df = pd.DataFrame({
    "x": embedding[:, 0],
    "y": embedding[:, 1],
    f"{word_type}": filtered_vocab
})

In [146]:
target_words = ["père", "mère", "rêver"]

embedding_df["group"] = embedding_df[word_type].isin(target_words).map({
    True: "Target",
    False: "Other words"
})

In [147]:
fig = px.scatter(
    embedding_df,
    x="x",
    y="y",
    color="group",
    hover_name=f"{word_type}",
    title="UMAP Visualization of French Word Co-occurrences",
    opacity=0.8,
)

fig.update_traces(
    marker=dict(size=5)
)

fig.update_layout(
    width=1000,
    height=800,
    template="plotly_white",
    xaxis_title="UMAP 1",
    yaxis_title="UMAP 2",
)

fig.show()

In [149]:
min_word_occurrences = 500

filtered_vocab = [
    word for word, count in word_counts.items()
    if count >= min_word_occurrences
]

print(f"Vocabulary after filtering: {len(filtered_vocab):,}")
# Original vocabulary indices of retained words
filtered_ids = [
    word_to_id[word]
    for word in filtered_vocab
]

# Filter rows AND columns
filtered_matrix = cooc_matrix[filtered_ids, :][:, filtered_ids]
filtered_matrix = filtered_matrix.astype("float32")

print("Original matrix:", cooc_matrix.shape)
print("Filtered matrix:", filtered_matrix.shape)


import umap

reducer = umap.UMAP(
    n_components=3,
    n_neighbors=25,
    min_dist=0.8,
    metric="cosine",
    n_jobs=-1,
    # random_state=42
)

embedding = reducer.fit_transform(filtered_matrix)
embedding = embedding - embedding.mean(axis=0)


import pandas as pd
import plotly.express as px

embedding_df = pd.DataFrame({
    "x": embedding[:, 0],
    "y": embedding[:, 1],
    "z": embedding[:, 2],
    f"{word_type}": filtered_vocab
})

Vocabulary after filtering: 2,606
Original matrix: (285868, 285868)
Filtered matrix: (2606, 2606)


/home/antoine/cours/DAUPHINE_2026-2027/.venv/lib/python3.12/site-packages/umap/umap_.py:4160: FutureWarning: compatibility_layout was not set, so UMAP is using the legacy (0.5.x) nearest-neighbor, initialization and layout pipeline. In a future release the default will change to compatibility_layout=False, which uses recursive initialization and the Adam optimizer and produces different embeddings. Set compatibility_layout=True to keep the current behavior and silence this warning, or compatibility_layout=False (or optimizer='adam') to adopt the new behavior now.
  self.fit(X, y, ensure_all_finite, **kwargs)


In [150]:
import numpy as np
import pandas as pd
import umap
from sklearn.preprocessing import normalize

# 1. Filter vocabulary and co-occurrence matrix
min_word_occurrences = 500
filtered_vocab = [w for w, c in word_counts.items() if c >= min_word_occurrences]
filtered_ids = [word_to_id[w] for w in filtered_vocab]
filtered_matrix = cooc_matrix[filtered_ids, :][:, filtered_ids].astype("float32")

print(f"Vocabulary after filtering: {len(filtered_vocab):,}")
print(f"Matrix shape: {cooc_matrix.shape} -> {filtered_matrix.shape}\n")

# 2. UMAP dimensionality reduction + Mean-centering
reducer = umap.UMAP(
    n_components=3,
    n_neighbors=25,
    min_dist=0.8,
    metric="cosine",
    n_jobs=-1,
)
embedding = reducer.fit_transform(filtered_matrix)
embedding -= embedding.mean(axis=0)

embedding_df = pd.DataFrame({
    "x": embedding[:, 0],
    "y": embedding[:, 1],
    "z": embedding[:, 2],
    word_type: filtered_vocab
})

# 3. Extract target words, normalize, and compute cosine similarity
target_words = ["père", "mère", "rêver"]
target_df = embedding_df[embedding_df[word_type].isin(target_words)]

words = target_df[word_type].values
vectors = target_df[["x", "y", "z"]].values
norm_vectors = normalize(vectors, norm="l2")

print("--- Centered & Normalized Vectors (3D) ---")
for word, vec in zip(words, norm_vectors):
    print(f"{word:<8}: {np.round(vec, 4)}")

sim_df = pd.DataFrame(norm_vectors @ norm_vectors.T, index=words, columns=words)

print("\n--- Cosine Similarity Matrix ---")
print(sim_df.round(4))

Vocabulary after filtering: 2,606
Matrix shape: (285868, 285868) -> (2606, 2606)



/home/antoine/cours/DAUPHINE_2026-2027/.venv/lib/python3.12/site-packages/umap/umap_.py:4160: FutureWarning: compatibility_layout was not set, so UMAP is using the legacy (0.5.x) nearest-neighbor, initialization and layout pipeline. In a future release the default will change to compatibility_layout=False, which uses recursive initialization and the Adam optimizer and produces different embeddings. Set compatibility_layout=True to keep the current behavior and silence this warning, or compatibility_layout=False (or optimizer='adam') to adopt the new behavior now.
  self.fit(X, y, ensure_all_finite, **kwargs)


--- Centered & Normalized Vectors (3D) ---
père    : [ 0.766   0.4404 -0.4682]
mère    : [ 0.822   0.3492 -0.4499]
rêver   : [-0.9407  0.274   0.2003]

--- Cosine Similarity Matrix ---
         père    mère   rêver
père   1.0000  0.9941 -0.6937
mère   0.9941  1.0000 -0.7676
rêver -0.6937 -0.7676  1.0000


In [151]:
sim_df

,père,mère,rêver
père,1.000000,0.994108,-0.693667
mère,0.994108,1.000000,-0.767610
rêver,-0.693667,-0.767610,1.000000


## Évaluer un modèle de similarité

In [171]:
import pandas as pd

similarity_benchmarks = {
    "MC": "https://raw.githubusercontent.com/siabar/Multilingual_Wordpairs/master/Gold_Standards/fr-mc.dataset",
    "RG": "https://raw.githubusercontent.com/siabar/Multilingual_Wordpairs/master/Gold_Standards/fr-rg.dataset",
    "SimLex": "https://raw.githubusercontent.com/siabar/Multilingual_Wordpairs/master/Gold_Standards/fr-simlex.dataset",
    "WS353": "https://raw.githubusercontent.com/siabar/Multilingual_Wordpairs/master/Gold_Standards/fr-ws353.dataset",
}

datasets = {}

for name, url in similarity_benchmarks.items():
    datasets[name] = pd.read_csv(
        url,
        sep=";",
        # header=True
    )

    datasets[name] = datasets[name][["word1", "word2", "score"]].copy()

    datasets[name]["word1"] = datasets[name]["word1"].str.lower()
    datasets[name]["word2"] = datasets[name]["word2"].str.lower()

    print(f"{name}: {datasets[name].shape}")

MC: (30, 3)
RG: (65, 3)
SimLex: (999, 3)
WS353: (353, 3)


In [172]:
from scipy.stats import spearmanr
from sklearn.metrics.pairwise import cosine_similarity

def evaluate_similarity(dataset, matrix, word_to_id):

    human_scores = []
    model_scores = []

    for _, row in dataset.iterrows():

        word1 = row["word1"]
        word2 = row["word2"]

        if word1 not in word_to_id or word2 not in word_to_id:
            continue

        id1 = word_to_id[word1]
        id2 = word_to_id[word2]

        similarity = cosine_similarity(
            matrix[id1],
            matrix[id2]
        )[0, 0]

        human_scores.append(row["score"])
        model_scores.append(float(similarity))

    correlation, p_value = spearmanr(
        human_scores,
        model_scores
    )

    coverage = len(human_scores) / len(dataset)

    return correlation, p_value, coverage

In [173]:
results = {}

for name, dataset in datasets.items():

    correlation, p_value, coverage = evaluate_similarity(
        dataset,
        cooc_matrix,
        word_to_id
    )

    results[name] = {
        "Spearman": correlation,
        "p-value": p_value,
        "Coverage": coverage
    }

results_df = pd.DataFrame(results).T

print(results_df.round(4))

        Spearman  p-value  Coverage
MC        0.4831   0.0068    1.0000
RG        0.5356   0.0000    0.9692
SimLex    0.0812   0.0170    0.8629
WS353     0.1068   0.0638    0.8555


In [169]:
import numpy as np

# Joint probabilities
total = cooc_matrix.sum()
P_wc = cooc_matrix.astype(float) / total

# Marginal probabilities
P_w = np.asarray(P_wc.sum(axis=1)).flatten()
P_c = np.asarray(P_wc.sum(axis=0)).flatten()

# Calculate PPMI for non-zero entries
ppmi = P_wc.tocoo(copy=True)

ppmi.data = np.log(
    ppmi.data / (P_w[ppmi.row] * P_c[ppmi.col])
)

# Keep only positive values
ppmi.data = np.maximum(ppmi.data, 0)

ppmi_matrix = ppmi.tocsr()

In [170]:
results = {}

for name, dataset in datasets.items():

    correlation, p_value, coverage = evaluate_similarity(
        dataset,
        ppmi_matrix,
        word_to_id
    )

    results[name] = {
        "Spearman": correlation,
        "p-value": p_value,
        "Coverage": coverage
    }

results_df = pd.DataFrame(results).T

print(results_df.round(4))

        Spearman  p-value  Coverage
MC        0.3799   0.0384    1.0000
RG        0.6654   0.0000    0.9692
SimLex    0.1907   0.0000    0.8629
WS353     0.2266   0.0001    0.8555


In [174]:
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(n_components=100, random_state=42)

embeddings = svd.fit_transform(ppmi_matrix)

print("Embedding dimensions:", embeddings.shape)

Embedding dimensions: (285868, 100)


In [176]:
from scipy.stats import spearmanr
from sklearn.metrics.pairwise import cosine_similarity

def evaluate_similarity(dataset, matrix, word_to_id):
    human_scores, model_scores = [], []

    for _, row in dataset.iterrows():
        word1, word2 = row["word1"], row["word2"]

        if word1 not in word_to_id or word2 not in word_to_id:
            continue

        id1 = word_to_id[word1]
        id2 = word_to_id[word2]

        similarity = cosine_similarity(
            matrix[id1:id1+1],
            matrix[id2:id2+1]
        )[0, 0]

        human_scores.append(row["score"])
        model_scores.append(float(similarity))

    correlation, p_value = spearmanr(human_scores, model_scores)
    coverage = len(human_scores) / len(dataset)

    return correlation, p_value, coverage

In [177]:
results_svd = {}

for name, dataset in datasets.items():

    correlation, p_value, coverage = evaluate_similarity(
        dataset,
        embeddings,
        word_to_id
    )

    results_svd[name] = {
        "Spearman": correlation,
        "p-value": p_value,
        "Coverage": coverage
    }

results_svd_df = pd.DataFrame(results_svd).T

print(results_svd_df.round(4))

        Spearman  p-value  Coverage
MC        0.2760   0.1399    1.0000
RG        0.5064   0.0000    0.9692
SimLex    0.1100   0.0012    0.8629
WS353     0.0547   0.3434    0.8555


In [ ]:
from sklearn.decomposition import TruncatedSVD

dimensions = [50, 100, 200, 300, 500]

svd_results = {}

for n in dimensions:

    svd = TruncatedSVD(
        n_components=n,
        random_state=42
    )

    embeddings = svd.fit_transform(ppmi_matrix)

    scores = {}

    for name, dataset in datasets.items():

        correlation, p_value, coverage = evaluate_similarity(
            dataset,
            embeddings,
            word_to_id
        )

        scores[name] = correlation

    svd_results[n] = scores

svd_results_df = pd.DataFrame(svd_results).T

print(svd_results_df.round(4))